# Этап 3, версия 1.6. Адрес без потери редких микрорайонов

Проверяем, поможет ли модели точный микрорайон и дата публикации. Используем тот же диапазон цен, те же групповые train/validation/test и те же границы из версии 1.5. Сравниваем текущий LightGBM, LightGBM с датой и CatBoost с датой. Проверяем одни и те же объявления validation; test не используем.

Дата публикации известна при оценке объявления. `tom` и дата снятия объявления не используются: они могут стать известны только позже.


In [ ]:
# Воспроизводим разбиение и исходную модель версии 1.5 без повторного показа таблиц.
import json
from pathlib import Path
from IPython.utils.capture import capture_output
previous = next((p / "notebooks" / "03_modeling_v1.5.ipynb"
                 for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "notebooks" / "03_modeling_v1.5.ipynb").exists()), None)
if previous is None:
    raise FileNotFoundError("Не найдена версия этапа 3 v1.5")
with capture_output():
    for cell in json.loads(previous.read_text(encoding="utf-8"))["cells"]:
        if cell["cell_type"] == "code":
            result = get_ipython().run_cell("".join(cell["source"]))
            if result.error_before_exec or result.error_in_exec:
                raise result.error_before_exec or result.error_in_exec
print("Train:", len(range_train), "Validation:", len(range_val), "Test не используется:", len(range_test))
print("Границы, TRY:", q01, q90)


## 1. Сравниваемые признаки

Площадь и остальные признаки получаем теми же правилами, что в версии 1.5. Добавляем `days_since_start` — число дней от первой даты в train до публикации объявления. Вычисление опорной даты использует только train. Для CatBoost категории передаются как текст; редкие микрорайоны не объединяются вручную. Числовые пропуски заполняются медианой своего train.


In [ ]:
# Создаём одинаковую таблицу признаков для моделей с датой.
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             median_absolute_error, mean_absolute_percentage_error, r2_score)
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import warnings

x_train, x_val = make_features(range_train, range_val)
origin = range_train.listing_date.min()
x_train["days_since_start"] = (range_train.listing_date - origin).dt.days
x_val["days_since_start"] = (range_val.listing_date - origin).dt.days
numeric = num + ["days_since_start"]
assert x_train[numeric].shape[1] == x_val[numeric].shape[1]
print("Период:", range_train.listing_date.min().date(), "—", range_train.listing_date.max().date())
print("Разных микрорайонов train:", x_train.neighborhood.nunique())


## 2. Обучение

LightGBM с датой использует тот же One-Hot-подход и настройки, что модель версии 1.5; меняется только один признак. CatBoost получает категории напрямую. Все модели обучаются только на train. Для честной проверки не подбираем число итераций по этой же validation и не используем test.


In [ ]:
# Вариант LightGBM: проверяем вклад даты при прежнем кодировании.
onehot = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat", Pipeline([
        ("fill", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50))
    ]), cat)
], sparse_threshold=.8)
lightgbm_date = Pipeline([
    ("encode", onehot),
    ("model", TransformedTargetRegressor(
        regressor=LGBMRegressor(random_state=42, verbosity=-1, n_jobs=4, num_leaves=15),
        func=np.log1p, inverse_func=np.expm1))
])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lightgbm_date.fit(x_train, range_train.price)
    pred_lightgbm_date = lightgbm_date.predict(x_val)
print("LightGBM с датой обучена")

# Вариант CatBoost: сохраняем детальные текстовые категории.
cat_train, cat_val = x_train[numeric + cat].copy(), x_val[numeric + cat].copy()
for column in cat:
    cat_train[column] = cat_train[column].fillna("Неизвестно").astype(str)
    cat_val[column] = cat_val[column].fillna("Неизвестно").astype(str)
medians = cat_train[numeric].median()
cat_train[numeric] = cat_train[numeric].fillna(medians).astype(float)
cat_val[numeric] = cat_val[numeric].fillna(medians).astype(float)
cat_model = CatBoostRegressor(iterations=600, depth=8, learning_rate=0.08,
                              loss_function="RMSE", random_seed=42,
                              thread_count=4, verbose=False, allow_writing_files=False)
cat_model.fit(cat_train, np.log1p(range_train.price), cat_features=cat)
pred_catboost = np.expm1(cat_model.predict(cat_val))
print("CatBoost обучена; test не использовался")


## 3. Метрики на общей validation

Сначала оцениваем все типы жилья в выбранном диапазоне. Затем на тех же предсказаниях отдельно рассматриваем квартиры с исходно известной площадью и без неё. Заполненная медианой площадь не считается известной. MAE, RMSE и MedAE — в TRY, MAPE — в процентах.


In [ ]:
# Одинаковые объекты и одна шкала метрик для всех трёх моделей.
predictions = {
    "LightGBM v1.5": range_pred,
    "LightGBM + дата": pred_lightgbm_date,
    "CatBoost + дата": pred_catboost,
}
area = pd.to_numeric(range_val["size"], errors="coerce")
groups = {
    "Все объекты": np.ones(len(range_val), dtype=bool),
    "Квартиры, площадь известна": (range_val.sub_type.eq("Daire") & area.ge(15)).to_numpy(),
    "Квартиры, площадь неизвестна": (range_val.sub_type.eq("Daire") & ~area.ge(15)).to_numpy(),
}
rows = []
y = range_val.price.to_numpy()
for group_name, mask in groups.items():
    for model_name, pred in predictions.items():
        rows.append({
            "Группа": group_name, "Модель": model_name, "Объектов": int(mask.sum()),
            "MAE, TRY": mean_absolute_error(y[mask], pred[mask]),
            "RMSE, TRY": np.sqrt(mean_squared_error(y[mask], pred[mask])),
            "MedAE, TRY": median_absolute_error(y[mask], pred[mask]),
            "MAPE, %": 100*mean_absolute_percentage_error(y[mask], pred[mask]),
            "R²": r2_score(y[mask], pred[mask]),
        })
comparison = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
for group_name in groups:
    print(group_name)
    display(comparison.loc[comparison["Группа"].eq(group_name)].drop(columns="Группа").round(3))


## 4. Интерпретация

На **тех же 40 073 объявлениях validation** CatBoost снизил MAE с 64 249 до **52 500 TRY** (на 11 750 TRY, или 18,3%) и повысил R² с 0,585 до **0,711**. MAPE уменьшилась с 23,06% до 18,59%. LightGBM с датой дал ровно те же предсказания и метрики, что версия без даты: при его текущих настройках добавление даты пользы не принесло.

На квартирах с известной площадью CatBoost получил MAE **47 846 TRY**, R² **0,712** против 59 396 TRY и 0,577 у LightGBM. При неизвестной площади — MAE **51 253 TRY**, R² **0,670** против 61 822 TRY и 0,529. Улучшение видно в обеих группах, но R² разных групп напрямую не сравниваем: у них различается разброс цен.

Нельзя приписать весь прирост одному только сохранению редких микрорайонов: одновременно изменился алгоритм и способ кодирования категорий. Для выяснения причины нужен дополнительный эксперимент CatBoost без даты или с тем же укрупнением категорий.

Даже лучший результат относится только к сегменту **64 500–750 000 TRY**, сформированному по известной целевой цене. Это не доказательство качества на всех продажах; при прогнозе нового объявления его цену заранее не знаем. Финальный test оставлен для окончательного решения.
